# Prerequisites

In [ ]:
# get data for labs (English version and original French version)
!wget -nc -O around_the_world_in_80_days.txt https://www.gutenberg.org/ebooks/103.txt.utf-8
!wget -nc -O le_tour_du_monde_en_80_jours.txt https://www.gutenberg.org/ebooks/46541.txt.utf-8

# 1. Word Count

In [ ]:
# start a spark session and create spark context for making rdd
from pyspark.sql import SparkSession
spark = SparkSession.builder \
    .appName("word_count") \
    .getOrCreate()

sc = spark.sparkContext

In [ ]:
import os

# local paths inside the Docker container ('/content/' was a Colab path)
EN_PATH = 'file://' + os.path.abspath('around_the_world_in_80_days.txt')
FR_PATH = 'file://' + os.path.abspath('le_tour_du_monde_en_80_jours.txt')

# Define the rdd
rdd = sc.textFile(EN_PATH)

In [ ]:
# view the first x lines of the rdd
rdd.take(20)

In [ ]:
# example lambda function
words = rdd.flatMap(lambda lines: lines.split(' '))

In [ ]:
# Note and explain the output of the below command
words

**Explanation.** Only the RDD description is printed
(`PythonRDD[..] at RDD at ...`). `flatMap` is a **transformation**: Spark is
lazy, so it only records the step in the lineage (DAG). Nothing is computed yet.

In [ ]:
# Note and explain the output of the following command, focusing on the
# difference with the above command
words.collect()

**Explanation.** `collect()` is an **action**: it triggers a job and returns
all elements to the driver as a Python list. The empty strings `''` come from
blank lines and repeated spaces (`split(' ')`); words still carry capitals and
punctuation. On big data, prefer `take(n)` to avoid overloading the driver.

In [ ]:
# nicer print
for w in words.collect():
    print(w)

In [ ]:
# Print first x words
words.take(20)

In [ ]:
# Use cell magic command to help understand what the rdd.flatMap function is
# doing in the next cell.
rdd.flatMap?

In [ ]:
# map vs flatMap on the first 3 non-empty lines
sample = sc.parallelize(rdd.filter(lambda l: l.strip()).take(3))
print('map     :', sample.map(lambda l: l.split(' ')).collect())
print('flatMap :', sample.flatMap(lambda l: l.split(' ')).collect())

**Explanation.** `map` returns one element per line (here, a list of words).
`flatMap` applies the function and then **flattens** the result, so each word
becomes its own element. The next `map` turns each word into a `(word, 1)`
pair, the key/value format needed by `reduceByKey`.

In [ ]:
# Initialize a word counter by creating a tuple with word and count of 1
words = rdd.flatMap(lambda lines: lines.split(' ')) \
                    .map(lambda word: (word, 1))

for w in words.take(30):  # take instead of collect to limit the output
    print(w)

In [ ]:
# a. count the occurence of each word
from operator import add

counts = words.reduceByKey(add)   # same as lambda a, b: a + b
counts.take(20)

`reduceByKey` sums the values per key and pre-aggregates inside each partition
before the shuffle, which makes it cheaper than `groupByKey`.

In [ ]:
# b. a common first step in text analysis, change all capital letters to
# lower case
lower_counts = (rdd
                .map(lambda line: line.lower())
                .flatMap(lambda line: line.split(' '))
                .map(lambda word: (word, 1))
                .reduceByKey(add))
lower_counts.take(20)

In [ ]:
# c. eliminate the stop words.
# English stop words (based on NLTK's list, no extra dependency)
STOPWORDS_EN = set('''
a about above after again against all am an and any are aren't as at be
because been before being below between both but by can can't cannot could
couldn't did didn't do does doesn't doing don't down during each few for from
further had hadn't has hasn't have haven't having he he'd he'll he's her here
here's hers herself him himself his how how's i i'd i'll i'm i've if in into
is isn't it it's its itself let's me more most mustn't my myself no nor not
of off on once only or other ought our ours ourselves out over own same shan't
she she'd she'll she's should shouldn't so some such than that that's the
their theirs them themselves then there there's these they they'd they'll
they're they've this those through to too under until up upon very was wasn't
we we'd we'll we're we've were weren't what what's when when's where where's
which while who who's whom why why's with won't would wouldn't you you'd
you'll you're you've your yours yourself yourselves
s t d ll m re ve o y said one will now just
'''.split())

# broadcast: the set is shipped once per executor instead of with every task
stop_en = sc.broadcast(STOPWORDS_EN)

no_stop = (rdd
           .map(lambda line: line.lower())
           .flatMap(lambda line: line.split(' '))
           .filter(lambda w: w not in stop_en.value)
           .map(lambda w: (w, 1))
           .reduceByKey(add))
no_stop.take(20)

In [ ]:
# d. sort in alphabetical order
no_stop.sortByKey().take(20)

The empty string and tokens starting with punctuation (`'"i'`, `'(the'`) come
first, which shows why step f is needed.

In [ ]:
# e. sort descending by word frequency
no_stop.sortBy(lambda wc: wc[1], ascending=False).take(20)

In [ ]:
# f. remove punctuations and blank spaces
import re

# a token is a sequence of letters (accents included, digits excluded);
# punctuation and empty strings disappear. Curly apostrophes are normalized:
# "fogg's" -> ["fogg", "s"] and "s" is a stop word.
TOKEN_RE = re.compile(r"[^\W\d_]+", re.UNICODE)

def tokenize(line):
    return TOKEN_RE.findall(line.replace('’', "'"))

clean = (rdd
         .map(lambda line: line.lower())
         .flatMap(tokenize)
         .filter(lambda w: w not in stop_en.value)
         .map(lambda w: (w, 1))
         .reduceByKey(add))
clean.sortBy(lambda wc: wc[1], ascending=False).take(20)

### Bonus: remove the Project Gutenberg header and license
They add English words ("project", "gutenberg"...) that bias the counts.
The book text sits between the `*** START OF` and `*** END OF` lines.

In [ ]:
def strip_gutenberg(text_rdd):
    """Keep only the lines between the START and END markers."""
    indexed = text_rdd.zipWithIndex()   # (line, line number)
    starts = indexed.filter(lambda x: x[0].startswith('*** START')) \
                    .map(lambda x: x[1]).collect()
    ends = indexed.filter(lambda x: x[0].startswith('*** END')) \
                  .map(lambda x: x[1]).collect()
    start = starts[0] if starts else -1
    end = ends[0] if ends else float('inf')
    return indexed.filter(lambda x: start < x[1] < end) \
                  .map(lambda x: x[0])

book_en = strip_gutenberg(rdd).cache()
print(rdd.count(), 'total lines ->', book_en.count(), 'book lines')

### Final function: all transformations chained

In [ ]:
def word_count(lines_rdd, stopwords_bc, by='frequency'):
    """Count words in an RDD of lines.

    by='frequency': most to least frequent (alphabetical on ties)
    by='alpha'    : alphabetical order
    """
    counts = (lines_rdd
              .map(lambda line: line.lower())                 # b. lower case
              .flatMap(tokenize)                              # f. punctuation
              .filter(lambda w: w not in stopwords_bc.value)  # c. stop words
              .map(lambda w: (w, 1))
              .reduceByKey(add))                              # a. count
    if by == 'alpha':
        return counts.sortByKey()                             # d. alpha sort
    return counts.sortBy(lambda wc: (-wc[1], wc[0]))          # e. freq sort

en_counts = word_count(book_en, stop_en).cache()
en_counts.take(25)

In [ ]:
word_count(book_en, stop_en, by='alpha').take(25)

# 2. What does the following cell block do?
Comment the code below line by line after the provided hash-tag. You should be
able to explain each line while respecting the pep8 style guide of 79
characters or less per line!

In [ ]:
# Create an RDD of tuples (name, age)
dataRDD = sc.parallelize([("Brooke", 20), ("Denny", 31), ("Jules", 30),
                          ("TD", 35), ("Brooke", 25)])

# Compute the average age per name
agesRDD = (dataRDD
           # (name, age) -> (name, (age, 1)): age plus a counter of 1
           .map(lambda x: (x[0], (x[1], 1)))
           # per name, sum the ages and sum the counters
           # -> (name, (total_age, count))
           .reduceByKey(lambda x, y: (x[0] + y[0], x[1] + y[1]))
           # total_age / count -> (name, average_age)
           .map(lambda x: (x[0], x[1][0] / x[1][1])))

# action that triggers the computation
agesRDD.collect()

It computes the **average age per name**: Brooke → (20 + 25) / 2 = 22.5.
An average is not associative, so the (sum, count) pair is reduced and the
division is done at the end.

## 3. Function timing.

- write a simple python timer function for seeing how quickly your rdd runs as
  written. change the order of the steps in order to make the rdd run as
  optimally as possible

Each run ends with an action (`collect`), otherwise only the lazy plan is
timed. The input is cached first, each variant is run 5 times, and all
results are checked to be identical.

In [ ]:
import time
import statistics
from functools import wraps


def timer(func):
    """Simple decorator printing a function's run time."""
    @wraps(func)
    def wrapper(*args, **kwargs):
        t0 = time.perf_counter()
        result = func(*args, **kwargs)
        print(f'{func.__name__}: {time.perf_counter() - t0:.3f} s')
        return result
    return wrapper


def benchmark(func, *args, repeat=5):
    """Run func `repeat` times, return (min, median, result)."""
    times = []
    for _ in range(repeat):
        t0 = time.perf_counter()
        result = func(*args)
        times.append(time.perf_counter() - t0)
    return min(times), statistics.median(times), result


@timer
def run_once():
    return word_count(book_en, stop_en).collect()

_ = run_once()

In [ ]:
# Every variant returns the same (word, count) list sorted by frequency,
# only the order of the chained operations changes.

def v1_lab_order(lines):
    """Lab order a->f: count first, clean afterwards (3 shuffles, sort
    before filtering)."""
    return (lines
            .flatMap(lambda l: l.split(' '))
            .map(lambda w: (w, 1))
            .reduceByKey(add)                               # shuffle 1
            .map(lambda wc: (wc[0].lower(), wc[1]))
            .flatMap(lambda wc: [(t, wc[1]) for t in tokenize(wc[0])])
            .reduceByKey(add)                               # shuffle 2
            .sortBy(lambda wc: (-wc[1], wc[0]))             # shuffle 3
            .filter(lambda wc: wc[0] not in stop_en.value)  # late filter
            .collect())


def v2_filter_after_count(lines):
    """Clean before counting, but stop words removed after the shuffle."""
    return (lines
            .map(lambda l: l.lower())
            .flatMap(tokenize)
            .map(lambda w: (w, 1))
            .reduceByKey(add)
            .filter(lambda wc: wc[0] not in stop_en.value)
            .sortBy(lambda wc: (-wc[1], wc[0]))
            .collect())


def v3_lower_per_word(lines):
    """lower() called on every word instead of every line."""
    return (lines
            .flatMap(tokenize)
            .map(lambda w: w.lower())
            .filter(lambda w: w not in stop_en.value)
            .map(lambda w: (w, 1))
            .reduceByKey(add)
            .sortBy(lambda wc: (-wc[1], wc[0]))
            .collect())


def v4_groupbykey(lines):
    """groupByKey instead of reduceByKey: no map-side aggregation."""
    return (lines
            .map(lambda l: l.lower())
            .flatMap(tokenize)
            .filter(lambda w: w not in stop_en.value)
            .map(lambda w: (w, 1))
            .groupByKey()
            .mapValues(sum)
            .sortBy(lambda wc: (-wc[1], wc[0]))
            .collect())


def v5_optimal(lines):
    """lower per line -> tokenize -> filter -> reduceByKey -> sort."""
    return (lines
            .map(lambda l: l.lower())
            .flatMap(tokenize)
            .filter(lambda w: w not in stop_en.value)
            .map(lambda w: (w, 1))
            .reduceByKey(add)
            .sortBy(lambda wc: (-wc[1], wc[0]))
            .collect())


def v6_optimal_mappartitions(lines):
    """Clean and pre-count each partition in pure Python (Counter):
    far fewer objects exchanged between the JVM and Python."""
    from collections import Counter

    def count_partition(it):
        stop = stop_en.value
        c = Counter(w for l in it for w in tokenize(l.lower())
                    if w not in stop)
        return iter(c.items())

    return (lines
            .mapPartitions(count_partition)
            .reduceByKey(add)
            .sortBy(lambda wc: (-wc[1], wc[0]))
            .collect())


variants = [v1_lab_order, v2_filter_after_count, v3_lower_per_word,
            v4_groupbykey, v5_optimal, v6_optimal_mappartitions]

In [ ]:
book_en.count()           # materialize the cache before timing
reference = v5_optimal(book_en)

results = []
for f in variants:
    best, med, res = benchmark(f, book_en, repeat=5)
    assert res == reference, f'{f.__name__} gives a different result'
    results.append((f.__name__, best, med))

print(f"{'variant':<28}{'min (s)':>10}{'median (s)':>13}")
for name, best, med in sorted(results, key=lambda r: r[1]):
    print(f'{name:<28}{best:>10.3f}{med:>13.3f}')

**Analysis.** Exact times depend on the machine; close variants are within
measurement noise. `v6` is clearly the fastest and `v1` (lab order) clearly
the slowest. Best practices:
- filter as early as possible, before any shuffle;
- normalize keys before aggregating (a single `reduceByKey`);
- prefer `reduceByKey` over `groupByKey`;
- sort last, on the smallest data;
- work per line or per partition rather than per word;
- `broadcast` lookup data, `cache` reused RDDs.

On a single book in local mode, Spark's fixed job overhead dominates; the
gaps grow on large data where the shuffle is the main cost.

## 4. Text Comparison

- perform eda on the original french version of the
  [book](https://www.gutenberg.org/ebooks/46541.txt.utf-8) and compare the two

In [ ]:
# French stop words (articles, pronouns, prepositions, auxiliaries...)
STOPWORDS_FR = set('''
a à ai aie aient aies ait alors as au aucun aura aurai auraient aurais aurait
aux avaient avais avait avec avez aviez avions avoir avons ayant ce ceci cela
celle celles celui ces cet cette ceux chaque comme d dans de des deux donc dont
du elle elles en encore est et étaient étais était été être eu eue eues eurent
eus eut eux fait faire fut furent ici il ils j je jusqu l la le les leur leurs
lui m ma mais me même mes moi mon n ne ni nos notre nous on ont ou où par pas
peu peut plus pour pouvait qu quand que quel quelle quelles quels qui s sa sans
se sera serait ses si sien soit son sont sous sur t ta te tes toi ton tous tout
toute toutes très tu un une vers vos votre vous y c ça là cet dit ceux lequel
laquelle lesquels avait aussi bien fois sans après avant entre
'''.split())
stop_fr = sc.broadcast(STOPWORDS_FR)

rdd_fr = sc.textFile(FR_PATH)
book_fr = strip_gutenberg(rdd_fr).cache()
rdd_fr.take(15)

In [ ]:
fr_counts = word_count(book_fr, stop_fr).cache()
fr_counts.take(25)

In [ ]:
def stats(lines, stopwords_bc, counts):
    """A few EDA metrics for a book."""
    tokens = lines.map(lambda l: l.lower()).flatMap(tokenize).cache()
    n_tokens = tokens.count()
    n_unique = tokens.distinct().count()
    n_sw = tokens.filter(lambda w: w in stopwords_bc.value).count()
    avg_len = tokens.map(len).mean()
    hapax = counts.filter(lambda wc: wc[1] == 1).count()
    tokens.unpersist()
    return {
        'book lines': lines.count(),
        'non-empty lines': lines.filter(lambda l: l.strip()).count(),
        'words (tokens)': n_tokens,
        'distinct words': n_unique,
        'lexical richness (distinct/total)': round(n_unique / n_tokens, 4),
        'stop word share': round(n_sw / n_tokens, 4),
        'average word length': round(avg_len, 2),
        'hapax (seen once, no stop words)': hapax,
    }

s_en = stats(book_en, stop_en, en_counts)
s_fr = stats(book_fr, stop_fr, fr_counts)

print(f"{'metric':<40}{'english':>12}{'french':>12}")
for k in s_en:
    print(f'{k:<40}{s_en[k]:>12}{s_fr[k]:>12}')

In [ ]:
# Top 20 side by side
top_en = en_counts.take(20)
top_fr = fr_counts.take(20)
print(f"{'#':>3}  {'english':<22}{'french':<22}")
for i, (e, f) in enumerate(zip(top_en, top_fr), 1):
    print(f'{i:>3}  {e[0]:<14}{e[1]:>6}  {f[0]:<14}{f[1]:>6}')

### Word count comparison query
A `fullOuterJoin` on the word compares both counts. Shared words are mostly
**proper nouns** (characters, places), which align the two versions.

In [ ]:
comparison = (en_counts
              .fullOuterJoin(fr_counts)          # (word, (n_en, n_fr))
              .mapValues(lambda v: (v[0] or 0, v[1] or 0)))

# words present in both versions, by total frequency
common = (comparison
          .filter(lambda x: x[1][0] > 0 and x[1][1] > 0)
          .sortBy(lambda x: -(x[1][0] + x[1][1])))

print('words shared by both versions:', common.count())
print(f"{'word':<16}{'EN':>6}{'FR':>6}{'FR/EN':>8}")
for w, (n_en, n_fr) in common.take(25):
    print(f'{w:<16}{n_en:>6}{n_fr:>6}{n_fr / n_en:>8.2f}')

In [ ]:
# Focus on main characters and places
names = ['fogg', 'passepartout', 'fix', 'aouda', 'london', 'londres',
         'bombay', 'calcutta', 'hong', 'kong', 'yokohama', 'san',
         'francisco', 'new', 'york', 'liverpool']
names_rdd = sc.parallelize([(n, None) for n in names])

(names_rdd
 .leftOuterJoin(comparison)                        # (name, (None, (en, fr)))
 .map(lambda x: (x[0],) + (x[1][1] or (0, 0)))
 .sortBy(lambda x: -(x[1] + x[2]))
 .collect())

### Same comparison with Spark SQL

In [ ]:
spark.createDataFrame(en_counts, ['word', 'n_en']) \
     .createOrReplaceTempView('en')
spark.createDataFrame(fr_counts, ['word', 'n_fr']) \
     .createOrReplaceTempView('fr')

spark.sql('''
    SELECT COALESCE(en.word, fr.word)            AS word,
           COALESCE(n_en, 0)                     AS n_en,
           COALESCE(n_fr, 0)                     AS n_fr,
           COALESCE(n_fr, 0) - COALESCE(n_en, 0) AS diff
    FROM en FULL OUTER JOIN fr ON en.word = fr.word
    WHERE n_en IS NOT NULL AND n_fr IS NOT NULL
    ORDER BY n_en + n_fr DESC
    LIMIT 20
''').show()

### Observations
*(adjust to the numbers you get)*

- Both versions have a similar word count; stop words make up about half of
  the tokens in each language.
- French has more distinct words (conjugations, gender/number agreement) and
  slightly longer words.
- Main characters (`fogg`, `passepartout`, `fix`, `aouda`) top both lists with
  close counts; gaps come from translation choices (pronouns, "Mr.", etc.).
- Limits: home-made stop word lists, no lemmatization (nltk/spacy would help).

In [ ]:
# Release resources
spark.stop()